# 01 - SMILES ve RDKit Mol Nesnesi

Bu notebookta RDKit'te molek?lle ?al??man?n ilk teknik ad?m?n? ??renece?iz:

**SMILES metni ? RDKit `Mol` nesnesi ? temel molek?l kontrol? ? canonical SMILES ? basit g?rselle?tirme**

Bu ?al??ma e?itim ama?l?d?r. Ger?ek veri indirme, final dataset se?imi, modelleme, train/test split, metric/evaluation, Macro F1, benchmark veya bilimsel sonu? ?retimi yap?lmaz.


## Bu notebookta ne ??renilecek?

Bu notebookun sonunda ?unu rahat?a s?yleyebilmelisin:

> RDKit'te molek?lle ?al??man?n ba?lang?? noktas? ?o?u zaman SMILES metnini `Mol` nesnesine ?evirmektir. `Mol` nesnesi yaln?zca bir string de?ildir; atom, ba?, halka ve kimyasal yap? bilgisini ta??yan hesaplanabilir bir temsildir.

??renece?imiz ana kavramlar:

- SMILES nedir?
- RDKit `Mol` nesnesi nedir?
- Ge?erli ve ge?ersiz SMILES fark? nas?l g?r?l?r?
- Canonical SMILES neden duplicate fark?ndal??? i?in yararl?d?r?
- Basit molek?l bilgileri nas?l okunur?
- Molek?l ?izimi kalite kontrol d???ncesine nas?l yard?m eder?


## Bu notebookta ne yap?lmayacak?

- Ger?ek veri indirilmeyecek.
- `data/raw/` veya `data/processed/` kullan?lmayacak.
- `nmrshiftdb2rawdata.nmredata.sd` dosyas? ana veri olarak okunmayacak.
- Model kurulmayacak.
- Train/test split yap?lmayacak.
- Metric, benchmark, skor veya sahte sonu? ?retilmeyecek.
- Final dataset se?imi, kaynak se?imi, ADR kabul? veya learning gate ge?i?i yap?lmayacak.


## ?nceki notebookla ba?lant?

`00_rdkit_projedeki_rolu.ipynb` notebookunda RDKit'in projedeki genel rol?n? g?rd?k: RDKit, molek?l yap?s?n? Python i?inde temsil etmeye ve kimyasal veriyi daha d?zenli d???nmeye yard?m eder.

Bu notebookta art?k ilk teknik temsile ge?iyoruz. Bir molek?l? metin olarak yazaca??z, RDKit'e okutaca??z ve RDKit'in bize verdi?i `Mol` nesnesinin ne ta??d???n? anlamaya ba?layaca??z.


## SMILES nedir?

SMILES (Simplified Molecular Input Line Entry System), molek?l yap?s?n? tek sat?rl?k metin olarak yazman?n yayg?n bir yoludur.

SMILES bir insan ?izimi de?ildir. Yani ka??ttaki yap? form?l?n?n ayn?s? gibi d???n?lmemelidir. Daha ?ok bilgisayar?n okuyabilece?i kompakt bir yap? temsilidir.

?rnekler:

- Ethanol: `CCO`
- Acetone: `CC(=O)C`
- Acetic acid: `CC(=O)O`
- Benzene: `c1ccccc1`
- Ethyl acetate: `CCOC(=O)C`
- Aniline: `Nc1ccccc1`

Bu yaz?mlar k?sa g?r?n?r, ama do?ru yorumland???nda atom ve ba? ili?kilerini temsil eder.


## RDKit Mol nesnesi nedir?

RDKit'te temel d?n???m ?o?u zaman ?udur:

```python
mol = Chem.MolFromSmiles("CCO")
```

`Chem.MolFromSmiles()` SMILES metnini okumaya ?al???r. Ba?ar?l? olursa bir `Mol` nesnesi d?nd?r?r. Ba?ar?s?z olursa genellikle `None` d?nd?r?r.

`Mol` nesnesi d?z bir string de?ildir. ??inde atomlar, ba?lar, halka bilgisi, aromatiklik gibi kimyasal yap? bilgisini ta??yan hesaplanabilir bir temsil vard?r.


In [ ]:
# RDKit bu ortamda kurulu olmayabilir.
# G?venli import kullan?yoruz: RDKit yoksa notebook tamamen ??kmesin.

RDKit_AVAILABLE = False
rdkit_import_error = None
Draw = None

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    try:
        from rdkit.Chem import Draw
    except Exception:
        Draw = None
    RDKit_AVAILABLE = True
except Exception as exc:
    rdkit_import_error = exc

if RDKit_AVAILABLE:
    print("RDKit bulundu. Teknik h?creler ?al??t?r?labilir.")
else:
    print("RDKit bu ortamda bulunamad?.")
    print("Notebook kavramsal olarak okunabilir; RDKit gerektiren h?creler g?venli ?ekilde atlanacak.")
    print(f"Import hatas?: {type(rdkit_import_error).__name__}: {rdkit_import_error}")


## Oyuncak molek?l listesi

?lk teknik notebooklarda k???k ve kolay kontrol edilebilir molek?llerle ?al???yoruz. Bu liste final proje dataset'i de?ildir; sadece SMILES ve `Mol` nesnesi kavramlar?n? ??renmek i?indir.

NMReDATA learning sample dosyas? (`nmrshiftdb2rawdata.nmredata.sd`) bu notebookta ana veri olarak kullan?lmaz. O dosya daha ileride, SDF/NMReDATA okuma ve debugging ??renimi i?in `learning sample / pilot example` stat?s?nde devreye girebilir.


In [ ]:
toy_molecules = [
    {"molecule_name": "ethanol", "smiles": "CCO", "expected_note": "basit alkol"},
    {"molecule_name": "acetone", "smiles": "CC(=O)C", "expected_note": "basit keton/karbonil"},
    {"molecule_name": "acetic_acid", "smiles": "CC(=O)O", "expected_note": "karboksilik asit"},
    {"molecule_name": "benzene", "smiles": "c1ccccc1", "expected_note": "aromatik halka"},
    {"molecule_name": "ethyl_acetate", "smiles": "CCOC(=O)C", "expected_note": "ester ?rne?i"},
    {"molecule_name": "aniline", "smiles": "Nc1ccccc1", "expected_note": "aromatik amin"},
    {"molecule_name": "cyclohexane", "smiles": "C1CCCCC1", "expected_note": "alifatik halka"},
    {"molecule_name": "invalid_broken_ring", "smiles": "C1CC", "expected_note": "bilerek ge?ersiz SMILES"},
]

def print_rows(rows, columns):
    """K???k ??retici tablolar? ek paket kullanmadan yazd?r?r."""
    widths = []
    for column in columns:
        values = [str(row.get(column, "")) for row in rows]
        widths.append(max([len(column)] + [len(value) for value in values]))

    header = " | ".join(column.ljust(width) for column, width in zip(columns, widths))
    line = "-+-".join("-" * width for width in widths)
    print(header)
    print(line)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(width) for column, width in zip(columns, widths)))

print_rows(toy_molecules, ["molecule_name", "smiles", "expected_note"])


## SMILES ? Mol d?n???m?

?imdi her SMILES metnini RDKit'e okutmay? deniyoruz. Ge?erli olanlar `Mol` nesnesine d?n???r. Ge?ersiz olanlar `None` d?nebilir.

`None` sonucu ?nemlidir: RDKit bu metni g?venilir bir molek?l yap?s? olarak okuyamam??t?r. Bu durum veri temizli?i ve hata kontrol? a??s?ndan ileride ?ok ?nemli hale gelir.


In [ ]:
parsed_molecules = []

if RDKit_AVAILABLE:
    for item in toy_molecules:
        mol = Chem.MolFromSmiles(item["smiles"])
        parsed_molecules.append({**item, "mol": mol})

    rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        rows.append({
            "molecule_name": item["molecule_name"],
            "smiles": item["smiles"],
            "mol_created": mol is not None,
            "message": "okundu" if mol is not None else "None d?nd?: SMILES kontrol edilmeli",
        })

    print_rows(rows, ["molecule_name", "smiles", "mol_created", "message"])
else:
    parsed_molecules = [{**item, "mol": None} for item in toy_molecules]
    print("RDKit olmad??? i?in SMILES -> Mol d?n???m? ?al??t?r?lmad?.")
    print("Kavramsal fikir: ge?erli SMILES bir Mol nesnesine d?n???r; ge?ersiz SMILES None d?nebilir.")


## Mol nesnesinin string olmad???n? g?rmek

SMILES metni bir string'dir. `Mol` nesnesi ise RDKit'in hesaplanabilir molek?l temsilidir.

Bir `Mol` nesnesinden atom say?s?, ba? say?s? veya canonical SMILES gibi bilgiler ?retilebilir. Bu y?zden `Mol`, yaln?zca metni saklayan bir kutu gibi de?il, kimyasal yap?y? temsil eden bir nesne gibi d???n?lmelidir.


In [ ]:
if RDKit_AVAILABLE:
    example = next(item for item in parsed_molecules if item["molecule_name"] == "ethanol")
    mol = example["mol"]

    print(f"SMILES de?eri: {example['smiles']}")
    print(f"SMILES Python tipi: {type(example['smiles']).__name__}")
    print(f"Mol Python tipi: {type(mol).__name__}")
    print(f"Atom say?s?: {mol.GetNumAtoms()}")
    print(f"Ba? say?s?: {mol.GetNumBonds()}")
else:
    print("RDKit olmad??? i?in Mol nesnesi tipi g?sterilmedi.")
    print("Ana fikir: SMILES string, Mol ise atom ve ba? bilgisi ta??yan RDKit nesnesidir.")


## Canonical SMILES

Ayn? molek?l farkl? SMILES yaz?mlar?yla ifade edilebilir. RDKit, molek?l? okuduktan sonra canonical SMILES ?retebilir. Bu, ayn? yap?n?n farkl? metin yaz?mlar?yla tekrar tekrar g?r?nmesini fark etmek i?in yararl? bir ilk kontrold?r.

Ancak canonical SMILES tek ba??na bilimsel veri temizli?i anlam?na gelmez. Duplicate, stereokimya, tuzlar, tautomerler, ?l??m ko?ullar? ve metadata gibi konular ayr?ca d???n?lmelidir.


In [ ]:
canonical_examples = [
    {"molecule_group": "ethanol", "input_smiles": "CCO"},
    {"molecule_group": "ethanol", "input_smiles": "OCC"},
    {"molecule_group": "ethanol", "input_smiles": "C(C)O"},
    {"molecule_group": "acetic_acid", "input_smiles": "CC(=O)O"},
    {"molecule_group": "acetic_acid", "input_smiles": "CC(O)=O"},
]

if RDKit_AVAILABLE:
    rows = []
    for item in canonical_examples:
        mol = Chem.MolFromSmiles(item["input_smiles"])
        rows.append({
            "molecule_group": item["molecule_group"],
            "input_smiles": item["input_smiles"],
            "canonical_smiles": Chem.MolToSmiles(mol) if mol is not None else "OKUNAMADI",
        })

    print_rows(rows, ["molecule_group", "input_smiles", "canonical_smiles"])
else:
    print("RDKit olmad??? i?in canonical SMILES ?rne?i ?al??t?r?lmad?.")
    print("Kavramsal fikir: farkl? yaz?mlar, RDKit taraf?ndan ayn? canonical forma indirgenebilir.")


## Temel molek?l bilgileri

Bir `Mol` nesnesinden basit bilgiler okunabilir:

- Atom say?s?
- Ba? say?s?
- A??r atom say?s?
- Molek?l form?l?
- Molek?l a??rl???

Bunlar descriptor kavram?na giri? niteli?indedir. Descriptor'lar ileride daha sistematik ele al?nacakt?r. Bu notebookta bu de?erler yaln?zca RDKit nesnesinin hesaplanabilir oldu?unu g?stermek i?in kullan?l?r.


In [ ]:
if RDKit_AVAILABLE:
    info_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        if mol is None:
            info_rows.append({
                "molecule_name": item["molecule_name"],
                "atom_count": "-",
                "bond_count": "-",
                "heavy_atom_count": "-",
                "formula": "-",
                "mol_weight": "-",
            })
            continue

        info_rows.append({
            "molecule_name": item["molecule_name"],
            "atom_count": mol.GetNumAtoms(),
            "bond_count": mol.GetNumBonds(),
            "heavy_atom_count": mol.GetNumHeavyAtoms(),
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "mol_weight": round(Descriptors.MolWt(mol), 2),
        })

    print_rows(info_rows, ["molecule_name", "atom_count", "bond_count", "heavy_atom_count", "formula", "mol_weight"])
else:
    print("RDKit olmad??? i?in temel molek?l bilgileri hesaplanmad?.")


## Basit g?rselle?tirme

RDKit molek?l ?izimi yapabilir. G?rselle?tirme ?zellikle kalite kontrol i?in yararl?d?r: SMILES'?n bekledi?imiz yap?ya kar??l?k gelip gelmedi?ini h?zl?ca fark etmemizi sa?lar.

Bir ?izim, tek ba??na bilimsel do?rulama de?ildir. Ama yanl?? SMILES, beklenmeyen halka, yanl?? fonksiyonel grup veya okunamayan molek?l gibi durumlar? fark etmeye yard?m eder.


In [ ]:
if RDKit_AVAILABLE and Draw is not None:
    valid_items = [item for item in parsed_molecules if item["mol"] is not None]
    mols = [item["mol"] for item in valid_items]
    legends = [item["molecule_name"] for item in valid_items]

    # Notebook aray?z?nde bu nesne molek?l grid'i olarak g?r?nt?lenir.
    grid_image = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 180), legends=legends)
    grid_image
elif RDKit_AVAILABLE:
    print("RDKit bulundu, ancak Draw mod?l? bu ortamda kullan?lamad?.")
else:
    print("RDKit olmad??? i?in molek?l ?izimi yap?lmad?.")
    print("Kavramsal fikir: ?izim, SMILES'?n beklenen kimyasal yap?ya kar??l?k gelip gelmedi?ini h?zl?ca kontrol etmeye yard?m eder.")


## ??C NMR projesiyle ba?lant?

Bu proje ??C NMR spektrumlar?ndan fonksiyonel grup tahmini yapmay? hedefler. B?yle bir projede yap? bilgisi ?nemlidir ??nk? fonksiyonel gruplar molek?l yap?s?nda tan?mlan?r.

SMILES ve `Mol` nesnesi ileride ?u i?ler i?in temel olu?turur:

- SMARTS ile fonksiyonel grup veya alt yap? arama.
- Descriptor ve fingerprint ?retmenin ne anlama geldi?ini ??renme.
- Molek?l tekrarlar? ve duplicate riskleri hakk?nda d???nme.
- SDF/NMReDATA gibi daha ger?ek?i dosyalarda molecule/property ba?lant?s?n? kalite kontrolle inceleme.

Bu notebookta modelleme yap?lmad?. Burada yaln?zca molek?l temsilinin ilk ad?m? ??reniliyor.


## Mini al??t?rmalar

A?a??daki al??t?rmalar? kendi yorumunla deneyebilirsin:

1. Oyuncak molek?l listesine 3 yeni SMILES ekle.
2. Bilerek ge?ersiz bir SMILES yaz ve RDKit'in nas?l davrand???n? yorumla.
3. Ayn? molek?l i?in iki farkl? SMILES yaz?m? bul ve canonical SMILES ??kt?s?n? kar??la?t?r.
4. Bir molek?l?n atom say?s? ve ba? say?s?n? kimyasal yap?s?yla ili?kilendirerek a??kla.

Bu al??t?rmalar e?itim ama?l?d?r. Veri kayna?? se?imi, modelleme veya bilimsel sonu? ?retimi de?ildir.


## Kapan?? ?zeti

Bu notebookta SMILES'?n molek?l? metin olarak temsil eden kompakt bir g?sterim oldu?unu g?rd?k. RDKit'in `Chem.MolFromSmiles()` fonksiyonu ile SMILES metnini `Mol` nesnesine ?evirmeye ?al??t???n? ??rendik. `Mol` nesnesinin yaln?zca bir string olmad???n?; atom, ba? ve yap? bilgisi ta??yan hesaplanabilir bir temsil oldu?unu vurgulad?k. Ge?ersiz SMILES ?rne?inde `None` sonucunun neden ?nemli oldu?unu g?rd?k. Canonical SMILES'?n ayn? molek?l?n farkl? yaz?mlar?n? kar??la?t?rmada yararl? olabilece?ini, ama tek ba??na bilimsel veri temizli?i anlam?na gelmedi?ini belirttik. Basit atom, ba?, form?l ve molek?l a??rl??? kontrollerinin RDKit nesnesini anlamaya yard?m etti?ini g?rd?k. G?rselle?tirmenin kalite kontrol d???ncesine destek olabilece?ini konu?tuk. B?t?n bunlar, ileride ??C NMR projesinde yap? bilgisini daha dikkatli kullanmaya haz?rl?k i?indir.


In [ ]:
notebook_boundary_check = {
    "uses_toy_molecules_only": True,
    "reads_nmredata_learning_sample": False,
    "downloads_data": False,
    "writes_files": False,
    "trains_model": False,
    "creates_train_test_split": False,
    "computes_metrics_or_benchmark": False,
    "selects_final_dataset": False,
    "accepts_adr": False,
}

for key, value in notebook_boundary_check.items():
    print(f"{key}: {value}")
